In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

In [5]:
def csv_downloader(URL:str, name:str, path:str)->pd.DataFrame:
    df = pd.read_csv(URL)
    df.to_csv(f'{path}/{name}.csv', index = False)
    print(f'The {name}.csv was successfully stored in {path}')
    return df

In [11]:
df_orders = csv_downloader(
    URL='https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/rfm/orders.csv',
    name='orders',
    path='../data/rfm'
)

The orders.csv was successfully stored in ../data/rfm


In [12]:
df_orders.head()

,customer_id,order_date,revenue
0,Mr. Brion Stark Sr.,2004-12-20,32
1,Ethyl Botsford,2005-05-02,36
2,Hosteen Jacobi,2004-03-06,116
3,Mr. Edw Frami,2006-03-15,99
4,Josef Lemke,2006-08-14,76


In [14]:
fig = px.histogram(data_frame = df_orders, x='revenue', title='Distribution of Revenue')
fig.show()

In [19]:
max_date = df_orders['order_date'].max()

In [16]:
df_orders = pd.read_csv('../data/rfm/orders.csv', parse_dates=['order_date'])

In [22]:
rfm = df_orders.groupby('customer_id').agg(
  {
    'order_date': lambda date: (max_date - date.max()).days,
    'customer_id': lambda num: len(num),
    'revenue': lambda rev: rev.sum()
  }
)

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.reset_index(inplace=True)
rfm.head()

,customer_id,Recency,Frequency,Monetary
0,Abbey O'Reilly DVM,204,6,472
1,Add Senger,139,3,340
2,Aden Lesch Sr.,193,4,405
3,Admiral Senger,131,5,448
4,Agness O'Keefe,89,9,843


In [27]:
some_range = [1,2,3,4,5]
example = pd.qcut(some_range, 3, labels=['Low', 'Medium', 'High'])

example

['Low', 'Low', 'Medium', 'High', 'High']
Categories (3, str): ['Low' < 'Medium' < 'High']

In [29]:
rfm['R'] = pd.qcut(rfm['Recency'], 4, ['1', '2', '3', '4'])
rfm['F'] = pd.qcut(rfm['Frequency'], 4, ['4', '3', '2', '1'])
rfm['M'] = pd.qcut(rfm['Monetary'], 4, ['4', '3', '2', '1'])

In [30]:
pd.qcut(rfm['Recency'], 4, ['1', '2', '3', '4'])

0      2
1      2
2      2
3      1
4      1
      ..
990    2
991    2
992    3
993    4
994    4
Name: Recency, Length: 995, dtype: category
Categories (4, str): ['1' < '2' < '3' < '4']

In [31]:
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M
0,Abbey O'Reilly DVM,204,6,472,2,2,2
1,Add Senger,139,3,340,2,4,3
2,Aden Lesch Sr.,193,4,405,2,3,3
3,Admiral Senger,131,5,448,1,3,2
4,Agness O'Keefe,89,9,843,1,1,1


In [32]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Recency", "Frequency", "Monetary")
)

fig.add_trace(
    go.Histogram(x=rfm["Recency"], nbinsx=30, name="Recency"),
    row=1,
    col=1
)

fig.add_trace(
    go.Histogram(x=rfm["Frequency"], nbinsx=30, name="Frequency"),
    row=1,
    col=2
)

fig.add_trace(
    go.Histogram(x=rfm["Monetary"], nbinsx=30, name="Monetary"),
    row=1,
    col=3
)

fig.update_layout(
    title="",
    showlegend=False
)

fig.update_xaxes(title_text="Recency", row=1, col=1)
fig.update_xaxes(title_text="Frequency", row=1, col=2)
fig.update_xaxes(title_text="Monetary", row=1, col=3)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=3)

fig.show()

In [33]:
rfm['RFM_Score'] = rfm.R.astype(int) + rfm.F.astype(int) + rfm.M.astype(int)
rfm['RFM_Segment'] = rfm.R.astype(str) + rfm.F.astype(str) + rfm.M.astype(str)

rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
0,Abbey O'Reilly DVM,204,6,472,2,2,2,6,222
1,Add Senger,139,3,340,2,4,3,9,243
2,Aden Lesch Sr.,193,4,405,2,3,3,8,233
3,Admiral Senger,131,5,448,1,3,2,6,132
4,Agness O'Keefe,89,9,843,1,1,1,3,111


In [34]:
rfm = rfm.sort_values('RFM_Segment', ascending=False)
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
654,Mr. Cletus Corwin,557,2,81,4,4,4,12,444
379,Giuseppe Tremblay,576,3,162,4,4,4,12,444
330,Epifanio Kozey,492,1,43,4,4,4,12,444
824,Pierre Stoltenberg,617,2,82,4,4,4,12,444
333,Errol Bogisich,683,1,81,4,4,4,12,444


In [35]:
rfm_segments = rfm.groupby('RFM_Segment').count().reset_index(level=0)
rfm_segments.head()

,RFM_Segment,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score
0,111,62,62,62,62,62,62,62,62
1,112,18,18,18,18,18,18,18,18
2,113,2,2,2,2,2,2,2,2
3,121,10,10,10,10,10,10,10,10
4,122,26,26,26,26,26,26,26,26


In [36]:
fig = px.bar(
    rfm_segments.sort_values('customer_id'),
    x='customer_id',
    y='RFM_Segment',
    orientation='h',
    title='Number of Customers by RFM Segment'
)

fig.update_layout(
    xaxis_title='Number of Customers',
    yaxis_title='RFM Segment'
)

fig.show()

In [37]:
def naming(df):
    if df['RFM_Score'] >= 9:
        return 'Can\'t Loose Them'
    elif ((df['RFM_Score'] >= 8) and (df['RFM_Score'] < 9)):
        return 'Champions'
    elif ((df['RFM_Score'] >= 7) and (df['RFM_Score'] < 8)):
        return 'Loyal/Commited'
    elif ((df['RFM_Score'] >= 6) and (df['RFM_Score'] < 7)):
        return 'Potential'
    elif ((df['RFM_Score'] >= 5) and (df['RFM_Score'] < 6)):
        return 'Promising'
    elif ((df['RFM_Score'] >= 4) and (df['RFM_Score'] < 5)):
        return 'Requires Attention'
    else:
        return 'Demands Activation'

In [38]:
rfm['Segment_Name'] = rfm.apply(naming, axis=1)
rfm.head(5)

,customer_id,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment,Segment_Name
654,Mr. Cletus Corwin,557,2,81,4,4,4,12,444,Can't Loose Them
379,Giuseppe Tremblay,576,3,162,4,4,4,12,444,Can't Loose Them
330,Epifanio Kozey,492,1,43,4,4,4,12,444,Can't Loose Them
824,Pierre Stoltenberg,617,2,82,4,4,4,12,444,Can't Loose Them
333,Errol Bogisich,683,1,81,4,4,4,12,444,Can't Loose Them


In [39]:
grouped_by = rfm.groupby('Segment_Name').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(1)

In [40]:
grouped_by.columns = ['RecencyMean', 'FrequencyMean', 'MonetaryMean', 'Count']
grouped_by = grouped_by.reset_index()

grouped_by

,Segment_Name,RecencyMean,FrequencyMean,MonetaryMean,Count
0,Can't Loose Them,410.6,3.1,271.5,399
1,Champions,275.5,4.7,429.3,147
2,Demands Activation,78.7,8.7,870.6,62
3,Loyal/Commited,212.0,5.0,489.6,114
4,Potential,228.9,6.0,586.8,93
5,Promising,190.7,6.8,664.6,94
6,Requires Attention,136.4,7.8,765.9,86


In [41]:
fig = px.treemap(
    grouped_by,
    path=['Segment_Name'],
    values='Count',
    color='MonetaryMean',
    hover_data=['RecencyMean', 'FrequencyMean', 'MonetaryMean'],
    title='RFM Segments'
)

fig.show()